> **Versión corregida (v2.12, ejecución verificada)** — generada a partir de `01_fuentes_y_carga_reinicio.ipynb (Drive/Colab Notebooks)` sin sobrescribirlo.
>
> Cambios aplicados:
> - Celda de inicialización sustituida por el bootstrap portable: las rutas fijas `/content/drive/MyDrive/PIDA-RL-Diabetes` ahora dependen de `PROJECT_ROOT`.
> - Renombrado semántico: el antiguo `PROJECT_ROOT` (copia del repositorio) pasa a `REPO_ROOT`; el antiguo `PERSISTENT_ROOT` (Drive) pasa a `PROJECT_ROOT` (se conserva `PERSISTENT_ROOT` como alias).
> - El clonado usa la rama `feature/entorno-capacidad-v2`; antes clonaba `main`, que no contiene `src/` V2.
>
> Rutas: `PROJECT_ROOT` = datos/modelos/resultados (Drive en Colab, o `PIDA_PROJECT_ROOT`); `REPO_ROOT` = código. Modo prueba: `PIDA_MODO_PRUEBA=1` (cohorte pequeña y pocos episodios).


# PIDA-RL Diabetes — 01. Fuentes y carga controlada

Este notebook localiza, valida y carga las fuentes públicas de ENSANUT utilizadas para el proyecto. **No crea la cohorte, no imputa valores, no entrena modelos y no modifica los archivos fuente.**

Los microdatos se mantienen fuera del repositorio Git. La trazabilidad de las fuentes se guarda como metadatos ligeros en Google Drive.

## 1. Inicialización de rutas

Esta celda es autosuficiente: reutiliza Drive si ya está montado y reconoce tanto `/content/drive` como `/content/gdrive`.

In [ ]:
# === Bootstrap portable: PROJECT_ROOT (corrección v2.12) ===
# PROJECT_ROOT = raíz persistente de datos, modelos y resultados.
# REPO_ROOT    = copia local del código (repositorio GitHub).
# Prioridad: variables de entorno PIDA_PROJECT_ROOT / PIDA_REPO_ROOT,
# después Google Colab + Drive y, por último, ejecución local.
from pathlib import Path
from datetime import datetime, timezone
import os
import sys
import json
import importlib
import importlib.util
import subprocess

PROJECT_NAME = 'pida-rl-datos-publicos'

REPO_URL = 'https://github.com/JssSalas/pida-rl-datos-publicos.git'
BRANCH = os.environ.get('PIDA_BRANCH', 'main')
PROJECT_FOLDER_NAME = 'PIDA-RL-Diabetes'
IN_COLAB = 'google.colab' in sys.modules

DRIVE_ROOT = None
if os.environ.get('PIDA_PROJECT_ROOT'):
    PROJECT_ROOT = Path(os.environ['PIDA_PROJECT_ROOT']).expanduser().resolve()
elif IN_COLAB:
    DRIVE_ROOT = next(
        (p for p in [Path('/content/drive'), Path('/content/gdrive')] if (p / 'MyDrive').is_dir()),
        None,
    )
    if DRIVE_ROOT is None:
        from google.colab import drive
        mount_point = Path('/content/drive')
        if mount_point.exists() and any(mount_point.iterdir()):
            mount_point = Path('/content/gdrive')
        drive.mount(str(mount_point), force_remount=False)
        DRIVE_ROOT = mount_point
    PROJECT_ROOT = DRIVE_ROOT / 'MyDrive' / PROJECT_FOLDER_NAME
else:
    PROJECT_ROOT = (Path.cwd() / PROJECT_FOLDER_NAME).resolve()
PROJECT_ROOT.mkdir(parents=True, exist_ok=True)


def _buscar_repo(inicio):
    for candidato in [inicio, *inicio.parents]:
        if (candidato / 'src' / 'environment' / 'diabetes_followup_env.py').is_file():
            return candidato
    return None


if os.environ.get('PIDA_REPO_ROOT'):
    REPO_ROOT = Path(os.environ['PIDA_REPO_ROOT']).expanduser().resolve()
else:
    REPO_ROOT = _buscar_repo(Path.cwd().resolve()) or (
        Path('/content/pida-rl-datos-publicos') if IN_COLAB
        else Path.cwd().resolve() / 'pida-rl-datos-publicos'
    )
if not REPO_ROOT.exists():
    subprocess.check_call(['git', 'clone', '--branch', BRANCH, '--single-branch', REPO_URL, str(REPO_ROOT)])
elif not (REPO_ROOT / 'src').is_dir():
    raise RuntimeError(f'REPO_ROOT no contiene la carpeta src: {REPO_ROOT}')
elif IN_COLAB and (REPO_ROOT / '.git').is_dir():
    # En Colab se alinea la copia temporal con la rama de trabajo (sin push ni commit).
    for args in (['fetch', 'origin', BRANCH], ['checkout', BRANCH], ['pull', '--ff-only', 'origin', BRANCH]):
        subprocess.run(['git', '-C', str(REPO_ROOT), *args], check=False)

SRC_ROOT = REPO_ROOT / 'src'
for ruta in (str(REPO_ROOT), str(SRC_ROOT)):
    if ruta not in sys.path:
        sys.path.insert(0, ruta)
os.chdir(REPO_ROOT)

# Estructura persistente común (antes: rutas fijas /content/drive/MyDrive/PIDA-RL-Diabetes).
PERSISTENT_ROOT = PROJECT_ROOT  # alias de compatibilidad con los notebooks originales
MYDRIVE_ROOT = PROJECT_ROOT.parent
DATA_ROOT = PROJECT_ROOT / 'data'
RAW_DATA_DIR = DATA_ROOT / 'raw'
INTERIM_DATA_DIR = DATA_ROOT / 'interim'
PROCESSED_DATA_DIR = DATA_ROOT / 'processed'
METADATA_DIR = DATA_ROOT / 'metadata'
SPECIFICATIONS_DIR = DATA_ROOT / 'specifications'
MODELS_DIR = PROJECT_ROOT / 'models'
RESULTS_DIR = PROJECT_ROOT / 'results'
COHORT_PATH = PROCESSED_DATA_DIR / 'cohorte_diabetes.csv'

# Modo prueba: cohorte pequeña y pocos episodios (PIDA_MODO_PRUEBA=1).
MODO_PRUEBA = os.environ.get('PIDA_MODO_PRUEBA', '0') == '1'
N_COHORTE_PRUEBA = int(os.environ.get('PIDA_N_COHORTE', '120'))
SEMILLA_MUESTRA_PRUEBA = 2026
RESULTS_V2_DIR = RESULTS_DIR / ('capacidad_v2_prueba' if MODO_PRUEBA else 'capacidad_v2')


def muestra_cohorte_prueba(cohorte, n=None, semilla=SEMILLA_MUESTRA_PRUEBA):
    """Devuelve (cohorte, factor). En modo prueba toma una submuestra
    estratificada por categoria_riesgo (bajo/medio/alto); no crea columnas."""
    n = N_COHORTE_PRUEBA if n is None else int(n)
    if not MODO_PRUEBA or len(cohorte) <= n:
        return cohorte, 1.0
    fraccion = n / len(cohorte)
    muestra = (
        cohorte.groupby('categoria_riesgo', group_keys=False)
        .sample(frac=fraccion, random_state=semilla)
        .sort_values('FOLIO_INT')
        .reset_index(drop=True)
    )
    return muestra, len(muestra) / len(cohorte)


def escalar_capacidad(capacidad, factor):
    """Mantiene la presión de capacidad (recursos/persona) al reducir la cohorte."""
    return capacidad if factor >= 1.0 else max(1, int(round(capacidad * factor)))

print('Entorno:', 'Google Colab' if IN_COLAB else 'local')
print('PROJECT_ROOT (datos/resultados):', PROJECT_ROOT)
print('REPO_ROOT (código):', REPO_ROOT)
print('Modo prueba:', MODO_PRUEBA)

for directory in [RAW_DATA_DIR, INTERIM_DATA_DIR, METADATA_DIR]:
    directory.mkdir(parents=True, exist_ok=True)
print('Datos crudos:', RAW_DATA_DIR)
print('Directorio actual:', Path.cwd())


## 2. Dependencias de lectura

`pyreadstat` permite leer archivos SPSS `.sav` junto con sus metadatos.

In [ ]:
import subprocess

try:
    import pyreadstat
except ImportError:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--quiet', 'pyreadstat'])
    import pyreadstat

import pandas as pd

print('pyreadstat:', pyreadstat.__version__)
print('pandas:', pd.__version__)


## 3. Localizar fuentes

Coloca los archivos ENSANUT preferentemente en `MyDrive/PIDA-RL-Diabetes/data/raw/`. Se soportan nombres alternativos y se hace una búsqueda controlada en `data/raw` e `data/interim`; no se explora todo Drive para evitar tomar archivos equivocados.

In [ ]:
SOURCE_SPECS = {
    'adultos': {
        'expected_names': ['AdultosENSANUT2022.sav', 'Adultos_ENSANUT_2022.sav'],
        'keywords': ['adult', 'ensanut', '2022'],
        'required': True
    },
    'antro_tension': {
        'expected_names': ['AntroTensionENSANUT2022.sav', 'Antro_Tension_ENSANUT_2022.sav'],
        'keywords': ['antro', 'tension'],
        'required': False
    },
    'sangre': {
        'expected_names': ['MuestrasSangreENSANUT2022.sav', 'Muestras_Sangre_ENSANUT_2022.sav'],
        'keywords': ['sangre'],
        'required': False
    }
}

SEARCH_DIRS = [RAW_DATA_DIR, INTERIM_DATA_DIR]
ALL_SAV_FILES = sorted({
    path for directory in SEARCH_DIRS if directory.exists()
    for path in directory.rglob('*.sav')
    if path.is_file()
})

def locate_source(spec):
    for expected_name in spec['expected_names']:
        matches = [path for path in ALL_SAV_FILES if path.name.lower() == expected_name.lower()]
        if matches:
            return matches[0]

    keywords = [keyword.lower() for keyword in spec['keywords']]
    matches = [
        path for path in ALL_SAV_FILES
        if all(keyword in path.name.lower() for keyword in keywords)
    ]
    return matches[0] if len(matches) == 1 else None

source_paths = {name: locate_source(spec) for name, spec in SOURCE_SPECS.items()}

print('Archivos .sav detectados:')
for path in ALL_SAV_FILES:
    print(' -', path)

print('\nResolución de fuentes:')
for name, path in source_paths.items():
    status = path if path is not None else 'NO ENCONTRADO'
    print(f' - {name}: {status}')

missing_required = [
    name for name, spec in SOURCE_SPECS.items()
    if spec['required'] and source_paths[name] is None
]

if missing_required:
    raise FileNotFoundError(
        'No se encontraron fuentes obligatorias: ' + ', '.join(missing_required) + '.\n'
        f'Coloca el archivo en: {RAW_DATA_DIR}'
    )


## 4. Cargar archivos y metadatos

La carga se realiza en memoria. La cohorte se construirá hasta el notebook 03, después de revisar calidad y diccionario de variables.

In [ ]:
frames = {}
metas = {}

for name, path in source_paths.items():
    if path is None:
        print(f'{name}: se omite porque no fue localizado.')
        continue

    df, meta = pyreadstat.read_sav(path, apply_value_formats=False)
    frames[name] = df
    metas[name] = meta
    print(f'{name}: {df.shape[0]:,} filas x {df.shape[1]:,} columnas')

df_adultos = frames['adultos']

# Alias de compatibilidad para los notebooks 02 y 03.
df_diabetes = df_adultos

print('\ndf_adultos y df_diabetes disponibles en memoria.')


## 5. Registro de trazabilidad

Se guarda un inventario de archivos, dimensiones y fecha de carga. No se guardan microdatos ni valores personales en este registro.

In [ ]:
source_inventory = []
for name, path in source_paths.items():
    item = {
        'source_key': name,
        'path': str(path) if path else None,
        'filename': path.name if path else None,
        'exists': bool(path and path.is_file()),
        'size_bytes': path.stat().st_size if path and path.is_file() else None,
        'rows': int(frames[name].shape[0]) if name in frames else None,
        'columns': int(frames[name].shape[1]) if name in frames else None,
        'loaded_at_utc': datetime.now(timezone.utc).isoformat()
    }
    source_inventory.append(item)

inventory_path = METADATA_DIR / 'source_inventory.json'
with inventory_path.open('w', encoding='utf-8') as file:
    json.dump(source_inventory, file, ensure_ascii=False, indent=2)

display(pd.DataFrame(source_inventory))
print('Inventario guardado en:', inventory_path)


## 6. Inspección inicial

Esta inspección sirve para confirmar que los archivos se leyeron correctamente. La selección definitiva de variables y la revisión de faltantes ocurren en los notebooks 02 y 03.

In [ ]:
for name, df in frames.items():
    print('\n' + '=' * 80)
    print(name.upper())
    print('Dimensiones:', df.shape)
    print('Primeras columnas:', df.columns.tolist()[:25])
    display(df.head(3))


## Resultado esperado

Al terminar deben existir en memoria `df_adultos`, `df_diabetes`, `frames` y `metas`; en Drive debe existir `data/metadata/source_inventory.json`. Continúa con `02_entendimiento_datos.ipynb`.